# 05 — XGBoost Global Demand Model: M5 Walmart Demand Intelligence

**Goal:** Train a single global XGBoost model across all 30,490 product-store
series and evaluate via walk-forward cross-validation. Demonstrate that
incorporating price, SNAP, and lag features closes the univariate signal
ceiling identified in notebooks 02 and 03 — the three months where both
SARIMA and Prophet failed regardless of parameterization.

**Inputs:**
- `../data/processed/features_train.parquet`
- `../data/processed/features_val.parquet`
- `../data/processed/feature_cols.pkl`

**Outputs:**
- `../data/processed/xgb_model.pkl`
- `../data/processed/xgb_predictions_val.parquet`
- `../data/processed/xgb_cv_results.csv`

**Evaluation strategy:** Walk-forward cross-validation with 3 expanding folds.
Optuna hyperparameter tuning on Folds 1 and 2. Fold 2 best parameters frozen
and applied to Fold 3 without modification. Fold 3 test window touched exactly
once. It is the final benchmark against SARIMA (22.22% MAPE) and Prophet
(24.25% MAPE) on the representative series.

> **Demand proxy reminder:** All features and targets are derived from observed
> sales, which serve as a proxy for true latent demand. Zero sales on a given
> day may reflect a stockout (unmet demand) or genuine demand absence, 
> indistinguishable without inventory data. All model outputs should be
> interpreted as demand approximations that support inventory decisions,
> not exact demand recovery.

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna
import pickle
import os
import time
import warnings

from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams['figure.figsize'] = (14, 5)
np.random.seed(42)

# ── Paths ─────────────────────────────────────────────────────────────────
RAW_DIR       = '../data/raw'
PROCESSED_DIR = '../data/processed'
REP_SERIES    = 'FOODS_3_163_CA_3_validation'

# ── Walk-forward fold boundaries ──────────────────────────────────────────
FOLDS = {
    'fold_1': {
        'train_start': '2011-02-01',
        'train_end':   '2013-01-31',
        'val_start':   '2013-02-01',
        'val_end':     '2014-01-31',
    },
    'fold_2': {
        'train_start': '2011-02-01',
        'train_end':   '2014-01-31',
        'val_start':   '2014-02-01',
        'val_end':     '2015-01-31',
    },
    'fold_3': {
        'train_start': '2011-02-01',
        'train_end':   '2015-01-31',
        'val_start':   '2015-02-01',
        'val_end':     '2016-01-31',
    },
}

# ── XGBoost constants ─────────────────────────────────────────────────────
EARLY_STOPPING_ROUNDS = 50
OPTUNA_TRIALS         = 50
TARGET_COL            = 'target'
ID_COL                = 'id'
DATE_COL              = 'date'

# ── Evaluation function ───────────────────────────────────────────────────
def evaluate(actual, predicted, label):
    """
    Compute RMSE, MAE, MAPE on original unit scale.
    Inputs must already be back-transformed with np.expm1.
    MAPE guard: replace zero actuals with 1e-9 to avoid division by zero
    on structural gap rows — these reflect demand censoring, not true zeros.
    """
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae  = mean_absolute_error(actual, predicted)
    mape = np.mean(
        np.abs((actual - predicted) / np.where(actual == 0, 1e-9, actual))
    ) * 100

    print(f'{label}')
    print(f'  RMSE: {rmse:>10.4f}')
    print(f'  MAE:  {mae:>10.4f}')
    print(f'  MAPE: {mape:>10.2f}%')
    print()
    return {'label': label, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

print('All imports successful.')
print(f'XGBoost version: {xgb.__version__}')
print(f'Optuna version:  {optuna.__version__}')